# Literature Benchmark Replication

This notebook runs a preliminary structured benchmark pass. It maps established stress-indicator families to simple empirical baselines using a local Bloomberg export.

The committed notebook is intentionally unexecuted so proprietary Bloomberg-derived tables are not published. Run it locally to generate ignored outputs under `data/processed/`.

## Benchmark Families

- VIX / implied volatility: VIX level and changes.
- Volatility term structure: VIX3M minus VIX and UX2 minus UX1.
- Option tail risk: SKEW and VVIX.
- Equity state: SPX momentum, realized volatility, and trailing drawdown.
- Macro-financial context: yield-curve slopes and DXY.

The purpose is not to prove a trading signal. It is to establish what known structured indicators already explain before adding the longer cross-asset option-implied panel.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
for candidate in [repo, *repo.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
        repo = candidate
        break
sys.path.insert(0, str(repo / "src"))

from cross_asset_stress.experiments.literature_benchmarks import run_literature_benchmark_replication

input_path = repo / "data" / "raw" / "Book1.xlsx"
result = run_literature_benchmark_replication(input_path=input_path, output_dir=repo / "data" / "processed")
panel = result["panel"]
panel.shape, panel["date"].min(), panel["date"].max()

## Literature Anchors

In [ ]:
result["literature_anchors"]

## Conditional Event-Rate Lifts

These are simple conditional base-rate checks: when an indicator is in stress territory, how much higher is the subsequent drawdown event rate than unconditional?

In [ ]:
result["conditional_rates"].round(4)

In [ ]:
import matplotlib.pyplot as plt

cond = result["conditional_rates"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=False)
for ax, target in zip(axes, ["event_dd5_h20", "event_dd10_h20"]):
    plot_df = cond.loc[cond["target"].eq(target)].sort_values("lift_vs_base")
    ax.barh(plot_df["indicator"], plot_df["lift_vs_base"], color="#4c78a8")
    ax.axvline(1.0, color="black", linewidth=1)
    ax.set_title(target)
    ax.set_xlabel("Conditional event-rate lift vs base")
plt.tight_layout()
plt.show()

## Feature-Group Holdout Benchmarks

Train before 2020 and test from 2020 onward. This is deliberately hard because it includes COVID and the 2022 inflation/rates regime.

In [ ]:
result["model_results"].round(4)

In [ ]:
models = result["model_results"].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, target in zip(axes, ["event_dd5_h20", "event_dd10_h20"]):
    plot_df = models.loc[models["target"].eq(target)].sort_values("brier_score", ascending=False)
    ax.barh(plot_df["feature_group"], plot_df["brier_score"], color="#f58518")
    ax.set_title(f"{target}: Brier score lower is better")
    ax.set_xlabel("Brier score")
plt.tight_layout()
plt.show()

## Case Notes

These snapshots are for qualitative interpretation. They should not be used as model inputs.

In [ ]:
result["case_table"].round(4)

## Top Holdout Risk Dates

Top risk dates from the VIX term-structure benchmark on the 2020+ holdout.

In [ ]:
result["top_risk_dates"].round(4)

## Preliminary Interpretation

- Known stress indicators should be treated as the benchmark any new cross-asset stress layer must beat.
- VIX term structure and VIX futures backwardation are expected to be stronger for severe near-term stress than SKEW alone.
- SKEW is still worth retaining as a tail-pricing variable, especially for pre-stress conditions where VIX is calm.
- A negative or weak structured benchmark result is informative: it sets the calibration and validation standard for later cross-asset experiments.

In [ ]:
from IPython.display import Markdown, display

cond = result["conditional_rates"]
models = result["model_results"]
best_dd5_indicator = cond.loc[cond["target"].eq("event_dd5_h20")].sort_values("lift_vs_base", ascending=False).iloc[0]
best_dd10_indicator = cond.loc[cond["target"].eq("event_dd10_h20")].sort_values("lift_vs_base", ascending=False).iloc[0]
best_dd5_model = models.loc[models["target"].eq("event_dd5_h20")].sort_values("brier_score").iloc[0]
best_dd10_model = models.loc[models["target"].eq("event_dd10_h20")].sort_values("brier_score").iloc[0]

display(Markdown(f"""
### Generated Readout

- For the 5% / 20-day drawdown target, the strongest single stress-state lift is **{best_dd5_indicator['indicator']}**, with conditional event-rate lift of **{best_dd5_indicator['lift_vs_base']:.2f}x**.
- For the 10% / 20-day drawdown target, the strongest single stress-state lift is **{best_dd10_indicator['indicator']}**, with conditional event-rate lift of **{best_dd10_indicator['lift_vs_base']:.2f}x**.
- On the 2020+ holdout, the best Brier-score model for the 5% target is **{best_dd5_model['feature_group']}** with Brier **{best_dd5_model['brier_score']:.4f}**.
- On the 2020+ holdout, the best Brier-score model for the 10% target is **{best_dd10_model['feature_group']}** with Brier **{best_dd10_model['brier_score']:.4f}**.

Preliminary interpretation: known volatility and term-structure benchmarks are meaningful comparators. The AR/IVM cross-asset layer should be judged on incremental calibration, event-level detection, and portfolio path improvement rather than narrative plausibility.
"""))